# Customer Churn Prediction Pipeline

End-to-end churn analysis of bank customer data, in four stages:

1. **Data quality first**: duplicates and missing values are ruled out *before* any analysis, so dirty data cannot silently distort every statistic downstream
2. **EDA on a stratified sample**: EDA compute stays bounded even if a much bigger batch arrives in the future; key findings: **age** and **balance** drive churn, especially around the median balance
3. **ML on the full dataset**: preprocessing and a Delta table partitioned by `Geography` → `zero_balance` (whether the customer's balance is 0) — no train/test split
4. **Model comparison in MLflow**: four model families tuned by a **k-fold cross-validated grid search** (`CrossValidator` + `ParamGridBuilder`); the best cross-validated run is selected as the final model

## Setup & configuration

Single source of truth: every import, path, schema role, and flow knob lives here as a **global**, and the rest of the notebook only reads them so that nothing is re-imported or redefined downstream.

* **Paths**: `RAW_DIR` (input `churn.csv`), `PARTITIONED_DIR` (Delta output for the ML branch), `GRAPHS_DIR` (EDA figures), `ML_EXPERIMENT` (MLflow runs)
* **Schema roles**: `ID_COLS` (identifiers, never features), `NUMERICAL_COLS`, `CATEGORICAL_COLS`, `LABEL`
* **Flow knobs**: `SPLIT_SEED` (one seed for the EDA sample *and* the cross-validator folds, so both are reproducible), `TARGET_SAMPLE_SIZE` (EDA cap), `K_FOLDS` (folds per cross-validated grid search)

In [0]:
# --- Imports: the ONLY import block in the notebook (spark is built-in) ---
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker
from datetime import date

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, RobustScaler
from pyspark.ml.stat import Correlation
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.classification import (
    LogisticRegression, NaiveBayes, RandomForestClassifier, GBTClassifier,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator,
)
import mlflow
from mlflow import MlflowClient

# --- Path layout ---
WORK_PATH = os.getcwd()
VOLUME_PATH = "/Volumes/workspace/default/customer_data"
RAW_DIR = f"{VOLUME_PATH}/raw"
PARTITIONED_DIR = f"{VOLUME_PATH}/partitioned"
GRAPHS_DIR = f"{WORK_PATH}/graphs"
ML_EXPERIMENT = f"{WORK_PATH}/customer_churn_experiments"
CHURN_CSV = f"{RAW_DIR}/churn.csv"

# --- Schema roles (referenced everywhere instead of re-listing columns) ---
ID_COLS = ["RowNumber", "CustomerId", "Surname"]  # identifiers: never features
NUMERICAL_COLS = ["CreditScore", "Age", "Balance", "EstimatedSalary", "NumOfProducts", "Tenure"]
CATEGORICAL_COLS = ["Geography", "Gender"]
LABEL = "Exited"  # churn target (1 = churned)

# --- Flow knobs ---
SPLIT_SEED = 69             # one seed for the EDA sample AND the cross-validator folds
TARGET_SAMPLE_SIZE = 10000  # EDA branch cap: bounds compute if a bigger batch arrives
K_FOLDS = 5                 # ML branch: folds per cross-validated grid search (CrossValidator)

# Create the volume directories if they don't already exist
# (mkdirs also creates intermediate/parent directories as needed)
for _dir in (RAW_DIR, PARTITIONED_DIR):
    dbutils.fs.mkdirs(_dir)

# Serverless Spark ML: CrossValidator caches DataFrames internally, which on
# serverless compute requires a UC volume path for that temp cache — without
# this the cv.fit() cells raise PySparkException (verified 2026-11-21).
SPARKML_TMP_DIR = f"{VOLUME_PATH}/sparkml_tmp"
dbutils.fs.mkdirs(SPARKML_TMP_DIR)
os.environ["SPARKML_TEMP_DFS_PATH"] = SPARKML_TMP_DIR

print("Directories ensured:", RAW_DIR, "|", PARTITIONED_DIR, "|", GRAPHS_DIR)

## Load raw data

Read `churn.csv` from the raw volume (header + schema inference, permissive mode), then check the row count, schema, and a first sample of records.

In [0]:
churn_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(CHURN_CSV)
)

print("Rows:", churn_csv.count())
churn_csv.printSchema()
display(churn_csv.limit(10))


## Data quality checks

**Why this comes first:** a duplicate or a missing value would silently distort every statistic and every model trained downstream — so the data is proven clean *before* any analysis touches it. Only clean data proceeds to EDA.

* Summary statistics for the numerical columns
* Duplicate row count
* Null/NaN counts per column, unpivoted into a clean vertical table

In [0]:
churn_csv.select(NUMERICAL_COLS).summary().display()

In [0]:
num_duplicates = churn_csv.count() - churn_csv.distinct().count()
print(f"There are {num_duplicates} duplicates in this dataset.")

In [0]:
float_cols = {c for c, dtype in churn_csv.dtypes if dtype in ("float", "double")}
cols = churn_csv.columns

# Compute counts horizontally first
agg_exprs = [
    F.count(
        F.when(F.col(c).isNull() | (F.isnan(c) if c in float_cols else F.lit(False)), c)
    ).alias(c)
    for c in cols
]

# Unpivot into a clean vertical table
stack_expr = ", ".join([f"'{c}', `{c}`" for c in cols])

churn_csv.select(agg_exprs) \
    .select(F.expr(f"stack({len(cols)}, {stack_expr}) as (column_name, null_count)")) \
    .show(len(cols), truncate=False)

## Distributions & stratified EDA sample

EDA deliberately runs on a **stratified random sample** (`TARGET_SAMPLE_SIZE` rows, same seed as the ML split) that preserves the churn/retain proportions. Today the batch is 10k rows so the sample is effectively the whole dataset but if a much bigger batch arrives tomorrow, EDA compute stays capped while the ML branch still always trains on the **full** data.

1. Build the sample, then histograms of the numerical features
2. First churn-vs-retain aggregates: churners are ~7 years older (44.8 vs 37.4) and carry ~18k more balance (91.1k vs 72.7k): **age** and **balance** emerge as the leading churn drivers

In [0]:
# TARGET_SAMPLE_SIZE / SPLIT_SEED come from the setup cell — no local redefinitions.

total_count = churn_csv.count()
counts_per_class = churn_csv.groupBy("Exited").count().collect()

sample_ratio = min(1.0, TARGET_SAMPLE_SIZE / total_count)
fractions = {row["Exited"]: sample_ratio for row in counts_per_class}

# Stratified sample using native expressions (rand + comparison) instead of
# sampleBy, which compiles to a Scala UDF filter that Photon cannot accelerate.
rand_col = "__sample_rand"
keep_row = F.lit(False)
for stratum, fraction in fractions.items():
    keep_row = keep_row | ((F.col("Exited") == stratum) & (F.col(rand_col) < fraction))

some_churn = (
    churn_csv
    .withColumn(rand_col, F.rand(seed=SPLIT_SEED))
    .where(keep_row)
    .drop(rand_col)
)

In [0]:
# Collect only the numerical columns to the driver as Row objects, then plot each in the grid
numerical_rows = some_churn.select(NUMERICAL_COLS).collect()

ncols = 2
nrows = (len(NUMERICAL_COLS) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 3.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLS):
    values = [r[col] for r in numerical_rows]
    axes[i].hist(values, bins=30, color="#2b5c8f", alpha=0.7)
    axes[i].set_title(col)
    axes[i].grid(axis="y", linestyle="--", alpha=0.3)

# Hide any unused grid slots
for j in range(len(NUMERICAL_COLS), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
some_churn.groupBy("Exited").agg(
    F.round(F.mean("Age"), 1).alias("avgAge"),
    F.round(F.mean("IsActiveMember") * 100, 1).alias("PctActiveMembers"),
    F.round(F.mean("HasCrCard") * 100, 1).alias("PctCrCardHolders"),
    F.round(F.mean("NumOfProducts"), 2).alias("avgNumOfProducts"),
    F.round(F.mean("Balance"), 2).alias("avgBalance"),
    F.round(F.mean("EstimatedSalary"), 2).alias("avgEstimatedSalary"),
    F.round(F.mean("Tenure"), 1).alias("avgTenure")
).show()

some_churn.groupBy("Geography", "Gender").agg(
    F.count("*").alias("TotalCustomers"),
    F.round(F.mean("Balance"),1).alias("avgBalance"),
    F.round(F.mean("CreditScore")).alias("avgScore").cast('int'),
    F.sum("Exited").alias("ChurnedCount")
).sort(F.desc('avgBalance')).show()


## Feature correlation

Pearson correlation heatmap over all encoded features (string columns indexed to numeric), with the ID columns excluded. Purpose: spot redundant predictors *before* modeling to confirm no feature is so correlated with the label `Exited` that it would trivially encode the answer (a leakage check).

In [0]:

# 1. Exclude ID / identifier columns (ID_COLS from the setup cell)
candidate_cols = [c for c in churn_csv.columns if c not in ID_COLS]

# 2. Encode categorical string columns (Geography, Gender) to numeric for correlation
df_encoded = churn_csv
feature_cols = []

for c, dtype in churn_csv.select(candidate_cols).dtypes:
    if dtype == "string":
        indexed_col = f"{c}_idx"
        indexer = StringIndexer(inputCol=c, outputCol=indexed_col)
        df_encoded = indexer.fit(df_encoded).transform(df_encoded)
        feature_cols.append(indexed_col)
    else:
        feature_cols.append(c)

# 3. Assemble features into a vector and compute Pearson correlation in Spark
assembler = VectorAssembler(inputCols=feature_cols, outputCol="corr_features")
vector_df = assembler.transform(df_encoded).select("corr_features")

corr_matrix = Correlation.corr(vector_df, "corr_features", method="pearson").collect()[0][0].toArray()

# 4. Clean display labels (strip '_idx')
display_labels = [c.replace("_idx", "") for c in feature_cols]
n = len(display_labels)

# 5. Plot Heatmap using pure Matplotlib
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

# Set tick labels
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(display_labels, rotation=45, ha="right")
ax.set_yticklabels(display_labels)

# Add correlation coefficients inside cells
for i in range(n):
    for j in range(n):
        val = corr_matrix[i, j]
        text_color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color, fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson Correlation")
ax.set_title("Feature Correlation Heatmap", fontsize=13, pad=12)

plt.tight_layout()
plt.show()

In [0]:
# 1. Bucket ages < 30 into '20' (18-30), >= 80 into '80' (80+), and the rest in 10-year intervals
age_bucket_df = (
    some_churn
    .withColumn(
        "age_bin_start",
        F.when(F.col("Age") < 30, 20)
         .when(F.col("Age") >= 80, 80)
         .otherwise((F.floor(F.col("Age") / 10) * 10).cast("int"))
    )
    .groupBy("age_bin_start")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("Exited") == 1, 1).otherwise(0)).alias("churned_count"),
        F.sum(F.when(F.col("Exited") == 0, 1).otherwise(0)).alias("retained_count"),
    )
    .withColumn("churn_pct", (F.col("churned_count") / F.col("total")) * 100)
    .withColumn("retained_pct", (F.col("retained_count") / F.col("total")) * 100)
    .orderBy("age_bin_start")
)

# 2. Collect as standard Python Row objects
rows = age_bucket_df.collect()

# Format edge labels as '18-30' and '80+'
def format_age_label(bin_start):
    if bin_start == 20:
        return "18-30"
    elif bin_start == 80:
        return "80+"
    return f"{bin_start}-{bin_start + 10}"

x_labels = [format_age_label(r["age_bin_start"]) for r in rows]
churn_pct = [r["churn_pct"] for r in rows]
retained_pct = [r["retained_pct"] for r in rows]
x_pos = range(len(x_labels))

# 3. 100% Stacked Bar Chart with Matplotlib
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.bar(
    x_pos,
    churn_pct,
    label="Churned (1)",
    color="red",
    alpha=0.5,
    width=0.65
)
ax.bar(
    x_pos,
    retained_pct,
    bottom=churn_pct,
    label="Retained (0)",
    color="green",
    alpha=0.5,
    width=0.65
)

# Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45)
ax.set_xlabel("Age Group")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)
ax.set_title("Retention vs. Churn Rate by Age Group (100% Stacked)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
# Save the figure directly into the graphs volume (direct write works on this compute)
plt.savefig(f"{GRAPHS_DIR}/churn_by_age_group.png")
plt.show()

## Churn drivers
### Visual analysis

The aggregates above pointed at **age** and **balance** — especially around the **median balance (~97k)** — as the main churn drivers. The figures below (saved to `GRAPHS_DIR`) confirm and sharpen those findings:

* Retention vs churn rate by **age group** (100% stacked bars) -> churn increases with age but decrease after ~50.
* Rolling churn rate by **balance**, split by member activity (windowed SQL) -> around the median balance, active and inactive members diverge sharply but overall active members churn 15% less on average
* Customer volume vs churn rate by number of products -> 3+ products churn almost universally
* Churn rate by geography and by gender (SQL aggregates) -> Germany (~32%) churns at roughly twice the rate of France/Spain

In [0]:
some_churn.createOrReplaceTempView("churn")

In [0]:
sql_query = """
    SELECT 
        Balance,
        IsActiveMember,
        ROUND(
            AVG(Exited) OVER (
                PARTITION BY IsActiveMember
                ORDER BY Balance
                ROWS BETWEEN 500 PRECEDING AND 500 FOLLOWING
            ) * 100.0, 
            2
        ) AS rolling_churn_rate
    FROM churn
    WHERE Balance > 0
    ORDER BY Balance
"""

# 1. Collect as a Python list of Row objects (no pandas)
rows = spark.sql(sql_query).collect()

# 2. Plot using list comprehensions to filter and extract columns
fig, ax = plt.subplots(figsize=(10, 5))

for active_status, label, color in [
    (0, "Inactive Member (0)", "#d95f02"),
    (1, "Active Member (1)", "#2b5c8f")
]:
    balances = [r["Balance"] * 1e-3 for r in rows if r["IsActiveMember"] == active_status]
    rates = [r["rolling_churn_rate"] for r in rows if r["IsActiveMember"] == active_status]
    
    ax.plot(
        balances, 
        rates, 
        label=label, 
        color=color, 
        linewidth=2
    )

ax.set_ylim(10, 40)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:.0f}k"))
ax.set_xlabel("Balance")
ax.set_ylabel("Churn Probability (%)")
ax.set_title("Continuous Churn Rate by Balance (Split by Member Activity)")
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/churn_rate_by_balance.png")
plt.show()

In [0]:
# Aggregate in Spark, collect Row objects for plotting (notebook convention:
# Spark computes, matplotlib draws plain Python lists — no pandas)
prod_rows = (
    some_churn.groupBy("NumOfProducts")
    .agg(
        F.count("*").alias("Total"),
        (F.mean("Exited") * 100).alias("ChurnRate")
    )
    .orderBy("NumOfProducts")
    .collect()
)

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# Bar chart: Total Customers
color = "#2b5c8f"
ax1.set_xlabel("Number of Products")
ax1.set_ylabel("Total Customers", color=color)
bars = ax1.bar(
    [r["NumOfProducts"] for r in prod_rows],
    [r["Total"] for r in prod_rows],
    color=color, alpha=0.6, width=0.5
)
ax1.tick_params(axis="y", labelcolor=color)

# Line chart: Churn Rate (%)
ax2 = ax1.twinx()
color = "#d95f02"
ax2.set_ylabel("Churn Rate (%)", color=color)
ax2.plot(
    [r["NumOfProducts"] for r in prod_rows],
    [r["ChurnRate"] for r in prod_rows],
    color=color, marker="o", linewidth=2.5
)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Customer Volume vs. Churn Rate by Number of Products")
plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/churn_by_products.png")
plt.show()

In [0]:
spark.sql("""
    SELECT 
        Geography, 
        COUNT(*) AS TotalCustomers, 
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Geography
    ORDER BY PercentLost DESC
""").show()

spark.sql("""
    SELECT 
        Gender,
        COUNT(*) AS TotalCustomers,
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Gender
    ORDER BY PercentLost DESC;
""").show()

## ML branch
### preprocessing → partitioned Delta (Geography × zero-balance) → model input

The EDA branch above works on `some_churn` (stratified sample, unpartitioned — samples stay small by design).
The ML branch below works on the **full** dataset: only sampling bounds EDA compute — a churn model is only as good as the data it trains on.

1. Preprocess (drop IDs, encode categoricals, assemble ALL predictors into one `features` vector) and keep **only** `features`, the label `Exited`, and the partition keys — the raw columns are not needed downstream
2. Write once as Delta with `partitionBy("Geography", "zero_balance")`: **country first, then balance group** — roughly a third of customers carry a zero balance, a distinctly different population that stays physically separated on disk
3. Read back; the cross-validated grid search consumes the **whole** table

No train/test split exists anymore: model selection is **k-fold cross-validation** (`K_FOLDS`), so every row is written once and `CrossValidator` carves the folds at training time.

In [0]:
ml_base = churn_csv.drop(*ID_COLS)


gender_indexer = StringIndexer(inputCol="Gender", outputCol="Gender_idx").fit(ml_base)
geo_indexer = StringIndexer(inputCol="Geography", outputCol="Geography_idx").fit(ml_base)

ml_features = geo_indexer.transform(gender_indexer.transform(ml_base))


FEATURE_EXCLUDE = {"Exited", "Geography", "Gender"}
feature_cols = [c for c in ml_features.columns if c not in FEATURE_EXCLUDE]

model_input = (
    VectorAssembler(inputCols=feature_cols, outputCol="features")
    .transform(ml_features)
    .withColumn("zero_balance", F.col("Balance") == 0)
    .select("Geography", "zero_balance", "Exited", "features")
)

print("feature_cols:", feature_cols)
model_input.printSchema()

In [0]:
(
    model_input.write
    .format("delta")
    .partitionBy("Geography", "zero_balance")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(PARTITIONED_DIR)
)
print(f"Wrote Delta table partitioned by Geography, then zero_balance, to {PARTITIONED_DIR}")

In [0]:
ml_data = spark.read.format("delta").load(PARTITIONED_DIR)

print("total rows:", ml_data.count())
display(ml_data.groupBy("Geography", "zero_balance").count().orderBy("Geography", "zero_balance"))

## Model training & experiment tracking

Four model families compete under **one MLflow experiment**, each tuned by a **k-fold cross-validated grid search** (`CrossValidator` + `ParamGridBuilder`, `K_FOLDS` folds, one shared seed for reproducibility). Each parameter combination is scored by its average validation-fold AUC-ROC; the best combination per family is kept. Runs get distinct auto-incrementing names (`run<N>_<family>_<date>`), so repeated sessions never overwrite or blur the experiment history.

1. Select/create the experiment; alias a compact frame carrying `features` + `Exited`
2. Helpers: `f1_macro` and `cv_grid_log_model` (grid-search, log best params + per-combination CV AUC-ROC + full-data metrics of the winner)
3. `train_models` loops over the candidate families, names each run, and returns the best fitted models + run IDs in `results`

Note: Spark's `CrossValidator` folds are random, not class-stratified — with ~20% churn over 10k rows every fold's class balance stays close to, but not identical with, the full data. On serverless compute, `CrossValidator` also needs `SPARKML_TEMP_DFS_PATH` (set in the setup cell) for its internal temp cache.

In [0]:
if mlflow.get_experiment_by_name(ML_EXPERIMENT) is None:
    mlflow.create_experiment(ML_EXPERIMENT)
EXPERIMENT = mlflow.set_experiment(ML_EXPERIMENT)
EXPERIMENT_ID = EXPERIMENT.experiment_id
print(f"Logging runs to experiment '{ML_EXPERIMENT}' (id: {EXPERIMENT_ID})")

In [0]:
ml_all = ml_data.select("features", "Exited").cache()

total_rows = ml_all.count()
churn_rate = ml_all.select(F.mean("Exited")).first()[0]

print(f"rows: {total_rows} | churn rate: {churn_rate:.3f}")

In [0]:
def f1_macro(preds, label_col="Exited"):
    """Macro-averaged F1: the unweighted mean of the per-class F1 scores."""
    cm = (preds.groupBy(F.col(label_col).alias("label"), F.col("prediction").alias("pred"))
               .count()
               .collect())
    counts = {(float(r["label"]), float(r["pred"])): r["count"] for r in cm}

    def _f1(c):
        tp = counts.get((float(c),     float(c)),     0)
        fp = counts.get((float(1 - c),  float(c)),     0)  # predicted c, true is other
        fn = counts.get((float(c),     float(1 - c)), 0)  # true c, predicted other
        return 2 * tp / (2 * tp + fp + fn) if tp > 0 else 0.0

    return (_f1(0) + _f1(1)) / 2.0

In [0]:
def cv_grid_log_model(
    pipeline,
    param_grid,
    data,
    run_name: str,
    model_family: str,
    label_col: str = "Exited"
):
    """
    Runs a k-fold cross-validated grid search over `param_grid` for `pipeline`,
    then logs to MLflow: the best parameter combination found, the average
    validation-fold AUC-ROC of every grid combination, and full-data metrics of
    the best model (an in-sample sanity reference — the honest generalization
    estimate is the CV average).
    """
    auc_eval = BinaryClassificationEvaluator(
        labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )
    acc_eval = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol="prediction", metricName="accuracy"
    )

    # K_FOLDS / SPLIT_SEED come from the setup cell
    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=param_grid,
        evaluator=auc_eval,
        numFolds=K_FOLDS,
        seed=SPLIT_SEED,
        parallelism=2,
    )

    with mlflow.start_run(run_name=run_name) as run:
        print(
            f"[{run_name}] starting grid search: "
            f"{len(param_grid)} combos x {K_FOLDS} folds..."
        )
        cv_model = cv.fit(data)
        mlflow.set_tag("model_family", model_family)

        best_model = cv_model.bestModel       # fitted PipelineModel
        fitted_model = best_model.stages[-1]  # the model itself, after its scaler

        # Every combination's average validation AUC-ROC across the folds
        for i, avg in enumerate(cv_model.avgMetrics):
            print(f"[{run_name}] grid combo {i + 1}/{len(param_grid)}: avg AUC-ROC = {avg:.4f}")
        best_idx = int(np.argmax(cv_model.avgMetrics))

        # Best value of each tuned parameter, read off the fitted model
        fitted_param_map = {k.name: v for k, v in fitted_model.extractParamMap().items()}
        tuned_names = {p.name for pm in param_grid for p in pm}
        params = {"k_folds": K_FOLDS, "n_grid_combos": len(param_grid)}
        for p_name in sorted(tuned_names):
            params[f"best_{p_name}"] = str(fitted_param_map[p_name])

        # Feature count: tree models expose numFeatures, pipelines may not
        n_features = getattr(fitted_model, "numFeatures", None)
        if n_features is None:
            first = data.select("features").first()
            n_features = len(first[0]) if first else None
        params["n_features"] = n_features
        params["full_rows"] = data.count()

        # Full-data (in-sample) metrics of the best model — sanity reference;
        # the generalization estimate is the CV average above.
        preds = best_model.transform(data)
        metrics = {
            "cv_auc_roc": cv_model.avgMetrics[best_idx],
            "full_auc_roc": auc_eval.evaluate(preds),
            "full_f1_macro": f1_macro(preds),
            "full_accuracy": acc_eval.evaluate(preds),
        }
        for name, value in metrics.items():
            print(f"[{run_name}] {name}: {value:.4f}")

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        run_id = run.info.run_id

    print(f"MLflow run recorded: {run_id}\n")
    return cv_model, run_id

In [0]:
def train_models(data, models, grids):
    """
    Cross-validated grid search for each pipeline in `models`, logged to MLflow.

    Parameters
    ----------
    data : DataFrame
        Full features + label frame consumed by CrossValidator.
    models : dict[str, Pipeline]
        Maps model family name to an unfitted Pipeline.
    grids : dict[str, list]
        Maps model family name to its ParamGridBuilder grid.

    Returns
    -------
    dict[str, dict]
        {family_name: {"cv_model": CrossValidatorModel, "model": best PipelineModel, "run_id": run_id}}
    """
    client = MlflowClient()
    print(f"Logging runs to experiment '{EXPERIMENT}' (id: {EXPERIMENT_ID})")

    # Count existing runs per model family from MLflow tags
    run_counts = {}
    for run in client.search_runs(experiment_ids=[EXPERIMENT_ID], max_results=50000):
        family = run.data.tags.get("model_family")
        if family is not None:
            run_counts[family] = run_counts.get(family, 0) + 1

    today = date.today()
    results = {}

    for name, pipeline in models.items():
        nr = run_counts.get(name, 0) + 1
        run_name = f"run{nr}_{name}_{today}"

        cv_model, run_id = cv_grid_log_model(
            pipeline=pipeline,
            param_grid=grids[name],
            data=data,
            run_name=run_name,
            model_family=name,
        )

        run_counts[name] = nr
        results[name] = {
            "cv_model": cv_model,
            "model": cv_model.bestModel,
            "run_id": run_id,
        }

    return results

### Model definitions & grid search spaces

Four candidates, each on the feature view that suits it:

* **Logistic Regression** on `scaled_features` (scale-sensitive): the `RobustScaler` sits **inside** its pipeline, so each CV fold re-fits the scaler on the training folds only; elastic-net regularization
* **Random Forest** and **Gradient Boosting** on the raw `features` vector therefore tree models are scale-invariant
* **Naive Bayes** (multinomial) on the raw non-negative `features`which is a deliberately simple baseline; it expects count-like features, so weak scores are expected

`MODELS` maps each family name to a `Pipeline` (the names double as the `model_family` tags in MLflow). `PARAM_GRIDS` holds one grid per family, each seeded with the **Spark defaults** and the **settings used before the CV redesign** — edit the lists to explore further. `train_models` runs the cross-validated grid search for every family and stores the best fitted models + run IDs in `results`.

In [0]:
# One candidate per family, wrapped in a Pipeline. LABEL comes from the setup
# cell; the dict keys are the model_family tags used to name runs in MLflow.
# LR is scale-sensitive: RobustScaler sits INSIDE its pipeline so CrossValidator
# re-fits it on every training fold (no fold statistics leak into validation).
# Tree models and NB are scale-invariant and read the raw `features` vector.
lr = LogisticRegression(featuresCol="scaled_features", labelCol=LABEL)
rf = RandomForestClassifier(featuresCol="features", labelCol=LABEL)
nb = NaiveBayes(featuresCol="features", labelCol=LABEL, modelType="multinomial")
gb = GBTClassifier(featuresCol="features", labelCol=LABEL)

MODELS = {
    "Naive Bayes": Pipeline(stages=[nb]),
    "Logistic Regression": Pipeline(stages=[
        RobustScaler(inputCol="features", outputCol="scaled_features"), lr,
    ]),
    "Random Forest": Pipeline(stages=[rf]),
    "Gradient Boosting": Pipeline(stages=[gb]),
}

# Grid search spaces: each list carries the Spark DEFAULT and the setting used
# BEFORE the CV redesign — edit the lists to explore further.
PARAM_GRIDS = {
    "Naive Bayes": ParamGridBuilder()
        .addGrid(nb.smoothing, [1.0, 0.75])        
        .build(),
    "Logistic Regression": ParamGridBuilder()
        .addGrid(lr.elasticNetParam, [0.0, 0.1, 0.2]) 
        .build(),
    "Random Forest": ParamGridBuilder()
        .addGrid(rf.numTrees, [20, 50, 100])  
        .addGrid(rf.maxDepth, [5, 4, 3])
        .build(),          
    "Gradient Boosting": ParamGridBuilder()
        .addGrid(gb.maxIter, [20, 50, 100])            
        .addGrid(gb.maxDepth, [5, 4, 3])             
        .build(),
}

In [0]:
# MODELS / PARAM_GRIDS are defined in the cell above; results holds the best
# fitted CrossValidatorModels + MLflow run IDs.
results = train_models(
    data=ml_all,
    models=MODELS,
    grids=PARAM_GRIDS,
)

### Final model selection

Every run recorded in the experiment is compared on **cross-validated AUC-ROC** — the average validation-fold score of the best grid combination. The best run per family is reported as the **evaluation input** — the notebook does not pick a winner automatically.

The final model is chosen by hand: set `CHOSEN_MODEL` in the cell below to the family you decide on after reviewing the table (currently **Gradient Boosting**). CV scores are comparable across families because every family is evaluated on the same folds (`SPLIT_SEED`); full-data metrics are logged only as an in-sample sanity reference.

In [0]:
# --- Final model selection (manual) ---
# The table below is the evaluation input: the best run per model family by
# cross-validated AUC-ROC. The winner is NOT picked automatically — review the
# table, then set CHOSEN_MODEL to the family you decide on (any key of MODELS).
client = MlflowClient()

best_per_family = {}
for run in client.search_runs(experiment_ids=[EXPERIMENT_ID], max_results=50000):
    family = run.data.tags.get("model_family")
    auc = run.data.metrics.get("cv_auc_roc")
    if family is None or auc is None:
        continue
    if family not in best_per_family or auc > best_per_family[family]["cv_auc_roc"]:
        best_per_family[family] = {
            "cv_auc_roc": auc,
            "full_f1_macro": run.data.metrics.get("full_f1_macro", float("nan")),
            "full_accuracy": run.data.metrics.get("full_accuracy", float("nan")),
            "run_name": run.info.run_name,
            "run_id": run.info.run_id,
        }

print("Best run per model family (by cross-validated AUC-ROC):")
for family, m in sorted(best_per_family.items(), key=lambda kv: -kv[1]["cv_auc_roc"]):
    print(
        f"  {family:<20} cv_auc_roc={m['cv_auc_roc']:.4f}  "
        f"full_f1_macro={m['full_f1_macro']:.4f}  "
        f"full_accuracy={m['full_accuracy']:.4f}  ({m['run_name']})"
    )

final_name = "Gradient Boosting"
final_model = results[final_name]["model"]
final_run_id = results[final_name]["run_id"]
print(f"\nSelected final model: {final_name} (MLflow run {final_run_id})")